# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Inspect available record sets and fields. All entities are referenced by their `@id`. For each record set, we display its `@id`, fields and columns.

In [ ]:
# List all record sets with their @ids
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name')}")
    print(f"  description: {rs.get('description')}")
    if 'fields' in rs:
        print("  Fields and their @ids:")
        for f in rs['fields']:
            print(f"    - {f.get('@id')} ({f.get('name')})")
    if 'columns' in rs:
        print("  Columns:")
        for col in rs['columns']:
            print(f"    - {col.get('@id')} ({col.get('name')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as shown above. The following code dynamically constructs DataFrames for each record set, referencing by `@id`.

In [ ]:
# Prepare list of record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"RecordSet @ids to load: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    # Load records for each record set
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for {record_set_id}")
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records for {record_set_id}")
    except Exception as e:
        print(f"Failed to load {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. Make sure to reference fields by their `@id`.

In [ ]:
# Pick a record set and field for EDA; update the string to match a field that exists in your dataset.
if dataframes:
    # Use the first available record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Try to auto-detect a likely numeric field for demonstration
    numeric_candidates = df.select_dtypes(include='number').columns
    if len(numeric_candidates):
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field for filtering: {numeric_field_id}")
    else:
        numeric_field_id = None

    # Filtering example
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype in ['int64', 'float64'] else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {{numeric_field_id}} > {{threshold}}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to auto-detect a categorical/grouping field
        group_candidates = df.select_dtypes(include='object').columns
        group_field = group_candidates[0] if len(group_candidates) else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped mean data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA in this record set.")
else:
    print("No dataframes were loaded; skipping EDA.")

## 5. Visualization
Visualize distributions or relationships using the record set data. For demonstration, we plot a histogram of the first numeric field (referenced by its `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to explore the FAIR² dataset, dynamically referencing each record set and field by their `@id`. We loaded record set data, applied filtering and normalization, and visualized distributions—all using the provided Croissant schema. You can further adapt this notebook to target particular fields or add your own analyses as needed.